# 16 Standard Curve: Bradford-style Protein Assay - Python

## Biochemistry question

How can a standard curve be used to estimate unknown protein concentrations from absorbance values in a synthetic Bradford-style assay?

This notebook uses synthetic data for learning. The unknown concentrations are practice estimates, not real protein measurements.


In [ ]:
import plotly.io as pio
pio.renderers.default = "notebook_connected"


## 1. Setup

We use a synthetic standard curve dataset with known protein standards and unknown samples. The helper functions are in `src/standard_curve.py`.


In [ ]:
from pathlib import Path
import sys

import pandas as pd
import plotly.express as px

PROJECT_ROOT = Path("..").resolve()
sys.path.append(str(PROJECT_ROOT))

from src.standard_curve import (
    estimate_unknown_concentrations,
    fit_linear_standard_curve,
    summarize_standard_curve,
    summarize_unknown_estimates,
)


## 2. Dataset Preview

Standards have known concentrations. Unknown samples have blank concentration values and measured absorbance values.


In [ ]:
data_path = PROJECT_ROOT / "data" / "standard_curves" / "bradford_standard_curve.csv"
df = pd.read_csv(data_path)
df.head()


In [ ]:
df.groupby("sample_type").size().reset_index(name="row_count")


## 3. Summarize the Standards

For each known concentration, calculate the mean absorbance and replicate variability.


In [ ]:
standard_summary = summarize_standard_curve(df)
standard_summary


## 4. Fit a Linear Standard Curve

A simple linear model is useful only across the standard range where the assay behaves approximately linearly.


In [ ]:
fit = fit_linear_standard_curve(df)
{key: round(value, 4) for key, value in fit.items()}


In [ ]:
fig = px.scatter(
    standard_summary,
    x="known_concentration_mg_ml",
    y="mean_absorbance",
    error_y="sem_absorbance",
    title="Synthetic Bradford-style Standard Curve",
    labels={
        "known_concentration_mg_ml": "Known concentration (mg/mL)",
        "mean_absorbance": "Mean absorbance at 595 nm",
    },
)
fig.add_scatter(
    x=standard_summary["known_concentration_mg_ml"],
    y=fit["slope"] * standard_summary["known_concentration_mg_ml"] + fit["intercept"],
    mode="lines",
    name="Linear fit",
)
# If this chart does not render in Jupyter, try: fig.show(renderer="iframe") or fig.show(renderer="browser")
fig.show()


## 5. Estimate Unknown Concentrations

Unknown sample absorbance values are converted back to concentration estimates using the fitted line.


In [ ]:
unknown_estimates = estimate_unknown_concentrations(df, fit)
unknown_estimates[["sample_id", "replicate", "absorbance_595", "estimated_concentration_mg_ml", "outside_standard_range"]]


In [ ]:
unknown_summary = summarize_unknown_estimates(unknown_estimates)
unknown_summary.round(3)


## 6. Check for Extrapolation

Estimates outside the standard range should be treated carefully. In a real lab setting, those samples would often be diluted or re-run within the linear range.


In [ ]:
unknown_summary[["unknown_id", "mean_estimated_concentration", "any_outside_standard_range"]]


## What You Should Notice

- Absorbance increases as known protein concentration increases.
- The fitted line gives a simple calibration relationship for this synthetic assay.
- Unknown samples can be estimated from the line, but estimates outside the standard range need review.
- A high `r_squared` does not remove the need to check linear range and assay quality.

## Interpretation Practice

- Which unknown sample has the highest estimated concentration?
- Which unknown sample, if any, falls outside the standard range?
- Why is extrapolation risky in a standard curve assay?
- What would you do in a real lab if an unknown sample was above the highest standard?

## Common Mistake

- Do not assume every absorbance value can be converted safely. Standard curves are most useful inside the tested standard range.

## Limitations

- This is synthetic learning data, not a real Bradford assay.
- The notebook uses a simple linear fit and does not model all assay chemistry or instrument behavior.
- Real unknowns may need dilution, re-measurement, blank correction, and protocol-specific review.
- These estimates should not be used for clinical, diagnostic, regulatory, or proprietary conclusions.
